<h1 style="color:DodgerBlue">Индивидальный проект</h1>

## Название проекта

Вариант 9. Система заказов товаров и услуг


## Описание проекта

Вариант 9. Программа хранит заказы и считает стоимость товаров.

Item — товар с названием и ценой. Order — общий класс заказа: номер, дата, сумма и список товаров. AddItem добавляет товар, RemoveItem удаляет, CalculateTotal складывает цены.

OnlineOrder хранит email и способ доставки. При добавлении товара выводит сведения о доставке. PhysicalOrder хранит адрес и при удалении товара сообщает о возврате. SpecializedOrder хранит специальные условия и считает сумму со скидкой.

Список товаров закрыт полем private — это инкапсуляция. Три вида заказов наследуются от Order. Методы virtual и override позволяют каждому виду заказа выполнять действие по-своему. Это показано в цикле по массиву Order[].

Добавлены конструкторы и свойства с get и set. Геттер цены возвращает price, сеттер принимает неотрицательную цену. Объект Customer взаимодействует с Order: метод Buy вызывает AddItem и передаёт объект Item.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [ ]:
using System;
using System.Collections.Generic;

public class Item
{
    public string Name { get; set; }
    private decimal price;
    public decimal Price
    {
        get { return price; }
        set { if (value >= 0) price = value; }
    }
    public Item(string name, decimal price)
    {
        Name = name;
        Price = price;
    }
}

public class Order
{
    public int OrderId { get; set; }
    public DateTime CreationDate { get; set; }
    public decimal TotalAmount { get; protected set; }
    private List<Item> items = new List<Item>();
    public Order(int id)
    {
        OrderId = id;
        CreationDate = DateTime.Today;
    }

    public virtual decimal CalculateTotal()
    {
        TotalAmount = 0;
        foreach (Item item in items)
            TotalAmount += item.Price;
        return TotalAmount;
    }

    public virtual void AddItem(Item item)
    {
        items.Add(item);
        CalculateTotal();
        Console.WriteLine("Добавлен товар: " + item.Name);
    }

    public virtual void RemoveItem(Item item)
    {
        if (items.Remove(item))
        {
            CalculateTotal();
            Console.WriteLine("Удалён товар: " + item.Name);
        }
        else
            Console.WriteLine("Такого товара нет в заказе");
    }

    protected bool HasItem(Item item)
    {
        return items.Contains(item);
    }
}

public class OnlineOrder : Order
{
    public string CustomerEmail { get; set; }
    public string DeliveryMethod { get; set; }
    public OnlineOrder(int id, string email, string delivery) : base(id)
    {
        CustomerEmail = email;
        DeliveryMethod = delivery;
    }

    public override void AddItem(Item item)
    {
        base.AddItem(item);
        Console.WriteLine("Доставка: " + DeliveryMethod);
        Console.WriteLine("Email: " + CustomerEmail);
    }

    public void ChangeDelivery(string method)
    {
        DeliveryMethod = method;
    }
}

public class PhysicalOrder : Order
{
    public string DeliveryAddress { get; set; }
    public PhysicalOrder(int id, string address) : base(id)
    {
        DeliveryAddress = address;
    }

    public override void RemoveItem(Item item)
    {
        bool found = HasItem(item);
        base.RemoveItem(item);
        if (found)
            Console.WriteLine("Возврат товара. Адрес доставки: " + DeliveryAddress);
    }

    public void ChangeAddress(string address)
    {
        DeliveryAddress = address;
    }
}

public class SpecializedOrder : Order
{
    public string SpecialConditions { get; private set; }
    private decimal discount;
    public SpecializedOrder(int id, decimal discount) : base(id)
    {
        SetDiscount(discount);
    }

    public void SetDiscount(decimal value)
    {
        if (value >= 0 && value <= 100)
        {
            discount = value;
            SpecialConditions = "Скидка " + value + "%";
            CalculateTotal();
        }
    }

    public override decimal CalculateTotal()
    {
        decimal sum = base.CalculateTotal();
        TotalAmount = sum - sum * discount / 100;
        return TotalAmount;
    }
}

public class Customer
{
    public string Name { get; set; }
    public Customer(string name)
    {
        Name = name;
    }
    public void Buy(Order order, Item item)
    {
        Console.WriteLine(Name + " покупает " + item.Name);
        order.AddItem(item);
    }
}

Item book = new Item("Книга", 500);
Item pen = new Item("Ручка", 100);
OnlineOrder online = new OnlineOrder(1, "student@example.com", "Курьер");
PhysicalOrder physical = new PhysicalOrder(2, "Тюмень, ул. Республики, 1");
SpecializedOrder special = new SpecializedOrder(3, 10);

Order[] orders = { online, physical, special };
foreach (Order order in orders)
{
    order.AddItem(book);
    order.AddItem(pen);
    Console.WriteLine("Заказ " + order.OrderId + ": " + order.CalculateTotal());
}
physical.RemoveItem(pen);
Console.WriteLine("После возврата: " + physical.CalculateTotal());
Console.WriteLine(special.SpecialConditions);

Customer customer = new Customer("Андрей");
Item pencil = new Item("Карандаш", 50);
customer.Buy(online, pencil);
Console.WriteLine("Сумма онлайн-заказа: " + online.CalculateTotal());
